In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

In [2]:
# 1. Load Data
train_path = '/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/analysis_data.csv'
score_path = '/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Data/scoring_data.csv'

analysis = pd.read_csv(train_path)
scoring = pd.read_csv(score_path)
scoring_ids = scoring['customer_id']

In [3]:
# 2. Feature Engineering (Replicating Sub4 Success)
# Apply to both training and scoring data
for df in [analysis, scoring]:
    # A. New Features
    df["txn_amount_proxy"] = df["num_transactions"] * df["avg_transaction_value"]
    df["online_share"] = df["online_shopping_freq"] / (df["num_transactions"] + 1)
    df["utilities_share"] = df["utility_payment_count"] / (df["num_transactions"] + 1)
    
    # B. Log Transform Skewed Features (List from Sub4)
    log_features = [
        "annual_income", "credit_limit", "num_transactions", 
        "avg_transaction_value", "online_shopping_freq", 
        "reward_points_balance", "travel_frequency", "utility_payment_count"
    ]
    
    for col in log_features:
        # Fill NaNs with median before logging to avoid errors, just in case
        if col in df.columns:
            # Shift by 1 to avoid log(0)
            df[col] = np.log1p(df[col].fillna(df[col].median()))

In [4]:
# 3. Separate Target and Features
target = "monthly_spend"
X = analysis.drop(columns=[target, 'customer_id'])
y = analysis[target]
X_score = scoring.drop(columns=['customer_id'])

In [5]:
# 4. Define Preprocessing Pipeline (From Sub4)
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "bool", "category"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

In [6]:
# 5. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=2101
)

In [7]:
# 6. Define Robust Random Forest Model
# "Robust" settings: high trees, limited depth, sqrt features to force diversity
rf_params = {
    'n_estimators': 500,
    'max_depth': 15,
    'max_features': 'sqrt',
    'min_samples_leaf': 4,
    'n_jobs': -1,
    'random_state': 2101
}

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(**rf_params))
])

print("Training Robust Random Forest...")
model_pipeline.fit(X_train, y_train)

Training Robust Random Forest...


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [8]:
# 7. Evaluation
test_pred = model_pipeline.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, test_pred))

print("-" * 30)
print(f"✅ Local Test RMSE (Random Forest): {rmse:.4f}")
print("-" * 30)

------------------------------
✅ Local Test RMSE (Random Forest): 272.1379
------------------------------


In [9]:
# 8. Generate Submission
final_pred = model_pipeline.predict(X_score)

submission = pd.DataFrame({
    'customer_id': scoring_ids,
    'monthly_spend': final_pred
})

file_name = 'Submission_Robust_RF.csv'
submission.to_csv(file_name, index=False)
print(f"🎉 Submission saved as: {file_name}")

🎉 Submission saved as: Submission_Robust_RF.csv


In [12]:
# Load your best submissions
# 1. Your best GB score (from Sub4 Branch)
sub_gb = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Codes/submission_gb2_final.csv')

In [13]:
# 2. The new Robust RF (from the code above)
sub_rf = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Codes/Submission_Robust_RF.csv')

In [14]:
# Verify IDs match
assert sub_gb['customer_id'].equals(sub_rf['customer_id']), "Customer IDs do not match!"

In [15]:
# Create Ensemble (Weighted Blend)
# We give slightly more weight (0.6) to your proven GB model, and 0.4 to the new RF
ensemble_pred = (0.6 * sub_gb['monthly_spend']) + (0.4 * sub_rf['monthly_spend'])

In [16]:
# Create Final Submission
submission_ensemble = pd.DataFrame({
    'customer_id': sub_gb['customer_id'],
    'monthly_spend': ensemble_pred
})

file_name_ens = 'Submission_Hybrid_Ensemble.csv'
submission_ensemble.to_csv(file_name_ens, index=False)
print(f"🎉 Ensemble Submission saved as: {file_name_ens}")

🎉 Ensemble Submission saved as: Submission_Hybrid_Ensemble.csv


In [18]:
# 1. Load your two best distinct submissions
# Ensure these files are in your working directory
sub_gb = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Codes/submission_gb2_final.csv')          # Best Score: 256.48
sub_rf = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Predictive Analytics/Assignments/PAC/Codes/Submission_Robust_RF.csv')    # Score: 259.32

# 2. Verify Alignment
assert sub_gb['customer_id'].equals(sub_rf['customer_id']), "IDs do not match!"

# 3. Create Weighted Ensemble
# Strategy: 70% Gradient Boosting (Stronger), 30% Random Forest (Stabilizer)
# This allows the GB to drive performance while RF corrects specific types of errors.
ensemble_pred = (0.7 * sub_gb['monthly_spend']) + (0.3 * sub_rf['monthly_spend'])

# 4. Create Submission File
submission_ensemble = pd.DataFrame({
    'customer_id': sub_gb['customer_id'],
    'monthly_spend': ensemble_pred
})

file_name_ens = 'Submission_Hybrid_Ensemble_7030.csv'
submission_ensemble.to_csv(file_name_ens, index=False)
print(f"🎉 Hybrid Ensemble Submission saved as: {file_name_ens}")
print(submission_ensemble.head())

🎉 Hybrid Ensemble Submission saved as: Submission_Hybrid_Ensemble_7030.csv
   customer_id  monthly_spend
0     20451981    1812.491315
1     23251656    2682.498267
2     64082260    1456.477418
3     82505923    2020.021429
4     89981994    1438.342642
